# A Harris fit across one current sheet

The Harris-sheet part of the former `SingleCSAnalysisBeta0.0625_HarrisFit` (its thickness part is the
`beta_p0.0625_per_component` case of `SingleCSAnalysis`). B is filtered component by component with iterative
filtering (Bz - 1 for the z component); along the steepest descent of $J^2$ through the chosen sheet, $|B_\perp|$ is
fitted with $B_0 |\tanh((s - s_0)/\delta)| + C$, and the component of $B_\perp$ along its direction at the start of the
cut with $B_0 \tanh(s/\delta) + C$.

Corrected here: the old notebook sampled $|B|$ over $\pm$100 cells but fitted it against $s$ from $-40$ to $40$, so its
$\delta$ came out 2.5 times too small; both fits now use the profile sampled over the fitted range, and are saved
under their own names.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pta import config
from pta.fields import current_density
from pta.filtering import iterative_filtering1D
from pta.plotting import save_figure, show_field, use_style
from pta.sheets import descent_direction, fit_harris, line_profile

use_style()
RESULTS = config.results_dir("single_sheet", "SingleCSHarrisFit")

In [ ]:
BETA_P = 0.0625
SNAPSHOT_TIME = 40
SHEET = (3731, 2294)                  # cell of the chosen J^2 maximum
ITERATIVE_INDEX = 1                   # filtered field = sum of the IMFs from this one on

J_CUT_HALF_LENGTH = 60                # cells
B_CUT_HALF_LENGTH = 100
HARRIS_FIT_HALF_LENGTH = 40           # the Harris fits use the cut from -40 to 40 cells
CUT_ZOOM_HALF_BOX = 500

In [ ]:
run = config.run("beta_scan", beta_p=BETA_P)
dx, dy = run.dx, run.dy
Bx, By, Bz = run.load_many(["Bx", "By", "Bz"], SNAPSHOT_TIME)
Bx, By, Bz = (iterative_filtering1D(component, ITERATIVE_INDEX) for component in (Bx, By, Bz - 1))
Jx, Jy, Jz = current_density(Bx, By, Bz, dx, dy)
J2 = Jx**2 + Jy**2 + Jz**2
B_perp = np.sqrt(Bx**2 + By**2)
direction = descent_direction(J2, SHEET)
when = rf"$\beta_p = {BETA_P}$, t = {SNAPSHOT_TIME}"

In [ ]:
# J^2 and |B_perp| along the cut
s, J2_profile = line_profile(J2, SHEET, direction, J_CUT_HALF_LENGTH)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(s * dx, J2_profile)
ax.axvline(0, linestyle="--")
ax.set_xlabel(r"Distance along the cut ($d_p$)")
ax.set_ylabel(r"$J^2$")
ax.set_title(r"$J^2$ along the steepest descent, " + when, fontsize=10)
save_figure(fig, RESULTS, "J2_cut")
plt.show()

s, Bx_profile = line_profile(Bx, SHEET, direction, B_CUT_HALF_LENGTH)
_, By_profile = line_profile(By, SHEET, direction, B_CUT_HALF_LENGTH)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(s * dx, np.hypot(Bx_profile, By_profile))
ax.axvline(0, linestyle="--")
ax.set_xlabel(r"Distance along the cut ($d_p$)")
ax.set_ylabel(r"$|B_\perp|$")
ax.set_title(r"$|B_\perp|$ along the steepest descent, " + when, fontsize=10)
save_figure(fig, RESULTS, "B_perp_cut")
plt.show()

# the cut on the map
fig, ax = plt.subplots(figsize=(6, 6))
i0, j0 = max(SHEET[0] - CUT_ZOOM_HALF_BOX, 0), max(SHEET[1] - CUT_ZOOM_HALF_BOX, 0)
show_field(ax, J2[i0:SHEET[0] + CUT_ZOOM_HALF_BOX, j0:SHEET[1] + CUT_ZOOM_HALF_BOX], dx, dy, label=r"$J^2$",
           corner=(i0 * dx, j0 * dy))
x, y = (SHEET[0] + 0.5) * dx, (SHEET[1] + 0.5) * dy
s, _ = line_profile(J2, SHEET, direction, J_CUT_HALF_LENGTH)
ax.plot(x + s * direction[0] * dx, y + s * direction[1] * dy, color="yellow", linewidth=2)
ax.scatter([x], [y], color="white", marker="x")
ax.set_title("The cut", fontsize=10)
save_figure(fig, RESULTS, "J2_cut_zoom")
plt.show()

In [ ]:
# Harris fits over the central part of the cut
s, Bx_profile = line_profile(Bx, SHEET, direction, HARRIS_FIT_HALF_LENGTH)
_, By_profile = line_profile(By, SHEET, direction, HARRIS_FIT_HALF_LENGTH)
B_perp_profile = np.hypot(Bx_profile, By_profile)
along_start = (Bx_profile[0] * Bx_profile + By_profile[0] * By_profile) / B_perp_profile[0]   # along B_perp at the start

fits = {
    "B_harris_fit_along_B": (along_start, fit_harris(s, along_start, centred=True),
                             r"$B_\perp$ along its direction at the start of the cut"),
    "B_harris_fit_magnitude": (B_perp_profile, fit_harris(s, B_perp_profile, magnitude=True), r"$|B_\perp|$"),
}
for name, (profile, fit, quantity) in fits.items():
    print(f"{name}: B0 = {fit['B0']:.4f} +/- {fit['B0_error']:.4f}, s0 = {fit['s0'] * dx:.3f} d_p, "
          f"half-thickness delta = {fit['delta'] * dx:.4f} +/- {fit['delta_error'] * dx:.4f} d_p, C = {fit['C']:.4f}")
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(s * dx, profile, label="Data")
    ax.plot(s * dx, fit["curve"], "--", label=rf"Harris fit ($\delta = {fit['delta'] * dx:.3f}\,d_p$)")
    ax.axvline(fit["s0"] * dx, linestyle=":", color="gray", alpha=0.6)
    ax.set_xlabel(r"Distance along the cut ($d_p$)")
    ax.set_ylabel(quantity)
    ax.set_title("Harris fit along the steepest descent, " + when, fontsize=9)
    ax.legend()
    save_figure(fig, RESULTS, name)
    plt.show()